# 面试问题：梯度累积、AMP Loss Scaling 与 Gradient Clipping 怎样正确组合？

可以直接复述的回答是：梯度累积要把每个 micro-batch 的 loss 除以累积步数，再依次 backward，才能等价于一个大 batch。AMP 中较小梯度可能在 FP16 下变成零，所以先放大 loss，再在更新前 unscale 梯度。梯度裁剪必须发生在 unscale 之后，否则阈值衡量的是放大后的假梯度。只有到 accumulation boundary 才能裁剪、更新和清梯度。动态 loss scale 遇到非有限梯度时跳过更新并降低 scale。下面用投诉升级分类器真实执行 forward/backward，逐个显示 micro-batch 梯度。

## 真实案例：八张客服工单是否需要立即升级

输入字段是等待时长、VIP 标记和支付失败标记，最后一条是等待 80 小时的极端工单。数据为脱敏结构化教学样本，不代表线上分布。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量与自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警以突出实验输出
torch.set_num_threads(1)  # 固定单线程执行
torch.manual_seed(8)  # 固定模型初始化
tickets = [  # 定义八张带业务语义的客服工单
    ("T-101", 1.0, 0.0, 0.0, 0.0),  # 短等待普通咨询无需升级
    ("T-102", 3.0, 1.0, 0.0, 0.0),  # VIP 但刚创建的工单
    ("T-103", 10.0, 0.0, 1.0, 1.0),  # 支付失败且长等待需要升级
    ("T-104", 6.0, 1.0, 1.0, 1.0),  # VIP 支付问题需要升级
    ("T-105", 2.0, 0.0, 1.0, 0.0),  # 新支付问题暂不升级
    ("T-106", 14.0, 0.0, 0.0, 1.0),  # 超时普通工单需要升级
    ("T-107", 8.0, 1.0, 0.0, 1.0),  # 长等待 VIP 工单需要升级
    ("T-108", 80.0, 1.0, 1.0, 1.0),  # 极端等待工单用于触发大梯度
]  # 结束八张工单
x = torch.tensor([[row[1] / 10.0, row[2], row[3]] for row in tickets], dtype=torch.float32)  # 缩放等待时长并形成特征张量
y = torch.tensor([row[4] for row in tickets], dtype=torch.float32)  # 形成二分类标签张量
print("输入预览：id | 等待小时 | VIP | 支付失败 | 升级标签")  # 输出原始工单字段标题
for row in tickets:  # 逐条展示八张工单
    print(f"{row[0]} | {row[1]:6.1f} | {int(row[2])} | {int(row[3])} | {int(row[4])}")  # 展示业务输入与监督标签
print("micro-batch 数=4，每批形状=(2, 3)")  # 明确本实验的累积结构

输入预览：id | 等待小时 | VIP | 支付失败 | 升级标签
T-101 |    1.0 | 0 | 0 | 0
T-102 |    3.0 | 1 | 0 | 0
T-103 |   10.0 | 0 | 1 | 1
T-104 |    6.0 | 1 | 1 | 1
T-105 |    2.0 | 0 | 1 | 0
T-106 |   14.0 | 0 | 0 | 1
T-107 |    8.0 | 1 | 0 | 1
T-108 |   80.0 | 1 | 1 | 1
micro-batch 数=4，每批形状=(2, 3)


## Baseline / 基线：累积时忘记除以 accumulation steps

四个 micro-batch 的平均 loss 若直接相加，梯度会变成大 batch 平均梯度的四倍。下面用同一初始参数比较错误累积与完整 batch 更新。

In [2]:
class TicketClassifier(torch.nn.Module):  # 定义最小线性工单分类器
    def __init__(self):  # 初始化可训练权重和偏置
        super().__init__()  # 初始化 PyTorch 模块基类
        self.weight = torch.nn.Parameter(torch.tensor([0.05, -0.05, 0.05]))  # 设置可复现初始特征权重
        self.bias = torch.nn.Parameter(torch.tensor(0.0))  # 设置可训练标量偏置
    def forward(self, features, use_fp16=True):  # 定义可选择半精度乘法的前向传播
        if use_fp16:  # 模拟 AMP 的低精度矩阵乘法分支
            return (features.half() @ self.weight.half()).float() + self.bias  # 半精度乘法后回到 FP32 计算损失
        return features @ self.weight + self.bias  # 提供纯 FP32 参考前向传播
def binary_cross_entropy(logits, labels):  # 手写数值稳定的二元交叉熵
    return (torch.clamp(logits, min=0.0) - logits * labels + torch.log1p(torch.exp(-logits.abs()))).mean()  # 避免直接计算对数概率溢出
initial_state = TicketClassifier().state_dict()  # 保存所有比较共享的初始参数
def one_accumulated_update(divide_loss, loss_scale=128.0, max_norm=1.0):  # 执行一次四 micro-batch 累积更新
    model = TicketClassifier()  # 创建独立模型防止实验互相污染
    model.load_state_dict(initial_state)  # 恢复公平比较的相同初值
    model.zero_grad()  # 清空更新前梯度
    trace = []  # 保存每个 micro-batch 的 loss 和原始梯度范数
    for micro_index in range(4):  # 依次处理四个两样本 micro-batch
        start = micro_index * 2  # 计算当前 micro-batch 起始位置
        logits = model(x[start:start + 2])  # 真实执行半精度前向传播
        loss = binary_cross_entropy(logits, y[start:start + 2])  # 计算当前小批量平均损失
        normalized_loss = loss / 4.0 if divide_loss else loss  # 正确方案除以累积步数而基线遗漏
        scaled_loss = normalized_loss * loss_scale  # 放大 loss 以保护低精度小梯度
        scaled_loss.backward()  # 真实执行 backward 并累加到参数梯度
        current_norm = float(torch.sqrt(sum(parameter.grad.float().square().sum() for parameter in model.parameters())))  # 计算仍处于放大状态的累计梯度范数
        trace.append((micro_index + 1, float(loss), current_norm))  # 保存逐 micro-batch 中间轨迹
    with torch.no_grad():  # 进入不记录梯度的更新阶段
        for parameter in model.parameters():  # 遍历权重和偏置参数
            parameter.grad.div_(loss_scale)  # 在裁剪前恢复真实梯度尺度
        unscaled_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in model.parameters()))  # 计算 unscale 后全局梯度范数
        clip_coefficient = min(1.0, max_norm / (float(unscaled_norm) + 1e-12))  # 手写全局范数裁剪系数
        for parameter in model.parameters():  # 遍历全部可训练参数
            parameter.grad.mul_(clip_coefficient)  # 按统一比例裁剪梯度方向不变
            parameter -= 0.3 * parameter.grad  # 在累积边界执行一次真实参数更新
    return model, trace, float(unscaled_norm), clip_coefficient  # 返回更新模型和完整梯度轨迹
bad_model, bad_trace, bad_unscaled_norm, bad_clip = one_accumulated_update(False)  # 执行遗漏 loss 除法的基线更新
print("micro | 原始 loss | 放大后的累计 grad norm")  # 输出错误累积过程表头
for micro, loss, norm in bad_trace:  # 逐批展示梯度累积过程
    print(f"{micro:5d} | {loss:9.5f} | {norm:12.4f}")  # 展示梯度随 micro-batch 不断增长
print(f"错误累积 unscaled norm={bad_unscaled_norm:.4f}，clip 系数={bad_clip:.4f}")  # 输出遗漏除法导致的梯度尺度

micro | 原始 loss | 放大后的累计 grad norm
    1 |   0.68573 |      71.9927
    2 |   0.66133 |      71.9975
    3 |   0.69118 |      78.9287
    4 |   0.60561 |     319.4436
错误累积 unscaled norm=2.4957，clip 系数=0.4007


## 核心实现：scale → backward 累积 → unscale → clip → step

正确累积与同一批八样本完整 batch 的梯度应一致。极端工单会触发裁剪，因此比较的是裁剪后的参数更新。

In [3]:
good_model, good_trace, good_unscaled_norm, good_clip = one_accumulated_update(True)  # 执行正确的累积、缩放和裁剪顺序
reference_model = TicketClassifier()  # 创建完整 batch 参考模型
reference_model.load_state_dict(initial_state)  # 恢复相同初始参数
reference_model.zero_grad()  # 清空参考模型梯度
reference_logits = reference_model(x)  # 对八条数据一次性执行同样的半精度前向
reference_loss = binary_cross_entropy(reference_logits, y)  # 计算完整 batch 平均损失
reference_loss.backward()  # 获取完整 batch 的真实梯度
with torch.no_grad():  # 执行参考模型全局范数裁剪与更新
    reference_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in reference_model.parameters()))  # 计算完整 batch 梯度范数
    reference_clip = min(1.0, 1.0 / (float(reference_norm) + 1e-12))  # 计算参考裁剪系数
    for parameter in reference_model.parameters():  # 遍历参考模型参数
        parameter.grad.mul_(reference_clip)  # 按相同阈值裁剪参考梯度
        parameter -= 0.3 * parameter.grad  # 执行与累积方案相同的学习率更新
good_vector = torch.cat([parameter.detach().flatten() for parameter in good_model.parameters()])  # 拼接正确累积后的全部参数
bad_vector = torch.cat([parameter.detach().flatten() for parameter in bad_model.parameters()])  # 拼接错误累积后的全部参数
reference_vector = torch.cat([parameter.detach().flatten() for parameter in reference_model.parameters()])  # 拼接完整 batch 参考参数
good_distance = float(torch.linalg.vector_norm(good_vector - reference_vector))  # 计算正确累积与参考更新距离
bad_distance = float(torch.linalg.vector_norm(bad_vector - reference_vector))  # 计算错误累积与参考更新距离
print("micro | loss | scaled accumulated grad norm")  # 输出正确累积中间过程表头
for micro, loss, norm in good_trace:  # 逐批展示正确累积轨迹
    print(f"{micro:5d} | {loss:7.5f} | {norm:12.4f}")  # 展示 loss 除法后的累计梯度
print(f"good norm={good_unscaled_norm:.4f}，clip={good_clip:.4f}，距 full batch={good_distance:.8f}")  # 展示正确方案与完整 batch 等价
print(f"bad  norm={bad_unscaled_norm:.4f}，clip={bad_clip:.4f}，距 full batch={bad_distance:.8f}")  # 展示错误方案偏离参考更新

micro | loss | scaled accumulated grad norm
    1 | 0.68573 |      17.9982
    2 | 0.66133 |      17.9994
    3 | 0.69118 |      19.7322
    4 | 0.60561 |      79.8609
good norm=0.6239，clip=1.0000，距 full batch=0.00005014
bad  norm=2.4957，clip=0.4007，距 full batch=0.11287115


## 逐样本结果：持续训练正确累积方案

In [4]:
trained_model = TicketClassifier()  # 创建用于多轮训练的分类器
trained_model.load_state_dict(initial_state)  # 从相同初始参数开始训练
training_losses = []  # 保存每轮完整数据损失
for epoch in range(30):  # 使用正确累积顺序训练三十轮
    trained_model.zero_grad()  # 在每个累积窗口开始时清空梯度
    for micro_index in range(4):  # 依次处理四个 micro-batch
        start = micro_index * 2  # 计算当前两样本切片起点
        micro_logits = trained_model(x[start:start + 2])  # 执行半精度矩阵乘法前向
        micro_loss = binary_cross_entropy(micro_logits, y[start:start + 2]) / 4.0  # 将小批损失按累积步数归一化
        (micro_loss * 128.0).backward()  # 放大归一化损失并累积梯度
    with torch.no_grad():  # 在累积边界执行 unscale、clip 和 step
        for parameter in trained_model.parameters():  # 遍历全部训练参数
            parameter.grad.div_(128.0)  # 恢复真实梯度尺度
        epoch_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in trained_model.parameters()))  # 计算本轮全局梯度范数
        epoch_clip = min(1.0, 1.0 / (float(epoch_norm) + 1e-12))  # 计算裁剪系数
        for parameter in trained_model.parameters():  # 逐参数应用裁剪后的更新
            parameter -= 0.3 * parameter.grad * epoch_clip  # 使用手写 SGD 更新参数
    with torch.no_grad():  # 关闭评估计算图
        epoch_loss = binary_cross_entropy(trained_model(x, use_fp16=False), y)  # 用 FP32 前向评估完整数据损失
    training_losses.append(float(epoch_loss))  # 保存本轮可比较损失
with torch.no_grad():  # 进入逐样本推理阶段
    probabilities = torch.sigmoid(trained_model(x, use_fp16=False))  # 计算八条工单升级概率
predictions = (probabilities >= 0.5).float()  # 使用固定阈值形成分类结果
training_accuracy = float((predictions == y).float().mean())  # 计算教学数据分类准确率
print("id | 标签 | 升级概率 | 预测")  # 输出逐样本结果表头
for index, ticket in enumerate(tickets):  # 遍历八张输入工单
    print(f"{ticket[0]} | {int(y[index])} | {probabilities[index]:8.4f} | {int(predictions[index])}")  # 展示真实 forward 推理结果
print(f"loss：初始后第1轮={training_losses[0]:.4f}，第30轮={training_losses[-1]:.4f}，accuracy={training_accuracy:.1%}")  # 展示优化趋势和最终指标

id | 标签 | 升级概率 | 预测
T-101 | 0 |   0.4486 | 0
T-102 | 0 |   0.5470 | 1
T-103 | 1 |   0.7498 | 1
T-104 | 1 |   0.6798 | 1
T-105 | 0 |   0.5278 | 1
T-106 | 1 |   0.8015 | 1
T-107 | 1 |   0.6910 | 1
T-108 | 1 |   0.9999 | 1
loss：初始后第1轮=0.5775，第30轮=0.4253，accuracy=75.0%


## 失败案例与修正：FP16 下溢、溢出与动态 scale

`1e-8` 直接转成 FP16 会变成零；先乘 65536 后可保存非零信息。相反，过大的梯度乘高 scale 会变成 `inf`，此时必须跳过参数更新并降低 scale。

In [5]:
tiny_gradient = torch.tensor([1e-8], dtype=torch.float32)  # 构造会在 FP16 下溢的小梯度
tiny_without_scale = tiny_gradient.half()  # 复现直接半精度存储变成零
tiny_scaled_storage = (tiny_gradient * 65536.0).half()  # 先放大再转半精度保存有效数值
tiny_recovered = tiny_scaled_storage.float() / 65536.0  # unscale 恢复近似原始梯度
large_gradient = torch.tensor([100.0], dtype=torch.float32)  # 构造可能被高 scale 放大的大梯度
overflow_storage = (large_gradient * 1024.0).half()  # 复现高 loss scale 导致 FP16 溢出
reduced_storage = (large_gradient * 128.0).half()  # 降低 scale 后重新表示同一梯度
should_skip_update = not bool(torch.isfinite(overflow_storage).all())  # 动态 scaling 检查非有限值并决定跳步
print(f"tiny：直接 FP16={float(tiny_without_scale[0])}，scale 后存储={float(tiny_scaled_storage[0])}，恢复={float(tiny_recovered[0]):.2e}")  # 展示 loss scaling 防下溢效果
print(f"large：scale=1024 得到 {float(overflow_storage[0])}，scale=128 得到 {float(reduced_storage[0])}")  # 展示溢出与降 scale 修正
print("检测到非有限梯度，是否跳过更新：", should_skip_update)  # 展示动态 loss scale 的门禁决策

tiny：直接 FP16=0.0，scale 后存储=0.0006551742553710938，恢复=1.00e-08
large：scale=1024 得到 inf，scale=128 得到 12800.0
检测到非有限梯度，是否跳过更新： True


## 结果解读

正确累积的参数几乎与完整 batch 完全相同；遗漏 `/4` 后 unscaled 梯度增大四倍，并改变裁剪后的有效更新。输出也说明 loss scaling 不是“让模型学得更快”，而是让 FP16 能存住小梯度；在 step 前必须还原尺度。

## 生产边界

本例在 CPU 上模拟 FP16 矩阵乘法和 master FP32 参数，没有 Tensor Core、分布式 all-reduce、梯度桶或真实 `GradScaler` 状态。生产中要在 accumulation boundary 同步梯度，非有限梯度时所有 rank 一致跳步，并把 scaler、优化器和累积位置写入 checkpoint。裁剪阈值需用验证集和梯度监控确定。

## 最小回归测试

In [6]:
assert len(tickets) >= 5  # 保证案例具有足够多业务样本
assert good_distance < 1e-4  # 保证正确累积与完整 batch 更新数值等价
assert bad_unscaled_norm > good_unscaled_norm * 3.9  # 保证遗漏 loss 除法真实放大约四倍梯度
assert good_clip <= 1.0 and good_unscaled_norm * good_clip <= 1.00001  # 保证裁剪发生在 unscale 后并满足阈值
assert float(tiny_without_scale[0]) == 0.0 and float(tiny_recovered[0]) > 0.0  # 保证小梯度下溢及 scaling 修正可复现
assert should_skip_update and torch.isfinite(reduced_storage).all()  # 保证动态 scale 能识别溢出并找到有限尺度
assert training_losses[-1] < training_losses[0] and training_accuracy >= 0.75  # 保证真实训练降低损失并达到基本分类效果